# Baseline — Seasonal Naive (lag-52)

Prediction = same store/dept sales 52 weeks earlier (fallbacks: lag 53, lag 51, series median, dept median).

This is the anchor: **any model scoring worse than this is broken, not 'still tuning'.** Also debugs the whole Kaggle submission pipeline on day one.

In [1]:
!pip install -q mlflow

import os
if not os.path.exists("walmart-sales-forecasting"):
    !git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git
import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")

import glob, shutil, zipfile
src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
assert src, "competition data not attached"
os.makedirs("data", exist_ok=True)
for f in glob.glob(src[0] + "/*"):
    (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
print("data/:", os.listdir("data"))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.6/954.6 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 12.5 MB/s eta 0:00:00
Cloning into 'walmart-sales-forecasting'...
re

In [2]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [3]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw("data")
print(train.shape, test.shape)

setup_mlflow("Baseline_Training")

(421570, 5) (115064, 4)
MLflow -> https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow | experiment: Baseline_Training


In [4]:
def seasonal_naive(history, target):
    hist = history[["Store", "Dept", "Date", "Weekly_Sales"]]
    out = target[["Store", "Dept", "Date"]].copy()
    for L in (52, 53, 51):
        lagged = hist.copy()
        lagged["Date"] = lagged["Date"] + pd.Timedelta(days=7 * L)
        out = out.merge(lagged.rename(columns={"Weekly_Sales": f"lag_{L}"}),
                        on=["Store", "Dept", "Date"], how="left")
    pred = out["lag_52"].fillna(out["lag_53"]).fillna(out["lag_51"])
    ser_med = hist.groupby(["Store", "Dept"])["Weekly_Sales"].median().rename("ser_med")
    dep_med = hist.groupby("Dept")["Weekly_Sales"].median().rename("dep_med")
    out = out.join(ser_med, on=["Store", "Dept"]).join(dep_med, on="Dept")
    pred = pred.fillna(out["ser_med"]).fillna(out["dep_med"]).fillna(0.0)
    return pred.to_numpy()

In [5]:
for fold in (1, 2):
    tr, va = split_fold(train, fold)
    rep = wmae_report(va["Weekly_Sales"], seasonal_naive(tr, va), va["IsHoliday"])
    with mlflow.start_run(run_name=f"SeasonalNaive_Fold{fold}"):
        mlflow.log_params({"model": "seasonal_naive_lag52", "fold": fold, **FOLDS[fold]})
        mlflow.log_metrics(rep)
    print(f"fold {fold}: {rep}")

🏃 View run SeasonalNaive_Fold1 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/0/runs/1c6f6c719aa744b9a6258c220ff8c95e
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/0
fold 1: {'wmae': 2031.217856254442, 'mae_holiday': 2300.0820836565094, 'mae_nonholiday': 1917.6506495305675}
🏃 View run SeasonalNaive_Fold2 at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/0/runs/b6e4e5022d6f4d2d8371d534a98cc1c3
🧪 View experiment at: https://dagshub.com/ekatsirekidze/walmart-sales-forecasting.mlflow/#/experiments/0
fold 2: {'wmae': 1802.858566967359, 'mae_holiday': 1813.4179948047597, 'mae_nonholiday': 1799.9846602384584}


In [6]:
# Kaggle submission (upload manually or: kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f ...)
sub = test.copy()
sub["pred"] = seasonal_naive(train, test)
make_submission(sub, "pred", "submission_baseline.csv")
print("wrote submission_baseline.csv")

wrote submission_baseline.csv
